# Notebook 03 — From demo to production: memory, pause, persist, parallel

**Workshop:** Agentic AI for Scientists — Week 3 (From Loops to Graphs)
**Block:** Production (23 min)
**Goal:** Take Week 2's ReAct agent — a `while` loop that forgot everything when the cell ended — and add the four things production needs: a clean **tool loop** (`ToolNode`), **memory across turns** (`MemorySaver`), a **pause for human approval** (`interrupt`), **durable** state (`SqliteSaver`), and a peek at **async parallel** fan-out. Each is one small additive step.


## 0. Setup

In [ ]:
%pip install -q \
    "langgraph>=1.2.4" "langgraph-checkpoint-sqlite>=3.1.0" \
    "langchain>=1.3.2" "langchain-core>=1.4.0" "langchain-google-genai>=4.2.4" python-dotenv

In [ ]:
import os

# Free Gemini API key (no credit card): https://aistudio.google.com/apikey
# Colab: add GOOGLE_API_KEY under the key icon (left sidebar) -> "Secrets".
# Local: put GOOGLE_API_KEY=AIza... in a .env file next to this notebook.
try:
    from google.colab import userdata  # type: ignore
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
except Exception:
    from dotenv import load_dotenv
    load_dotenv()

# Accept GEMINI_API_KEY as an alias (some AI Studio users store it under that name).
if not os.environ.get("GOOGLE_API_KEY") and os.environ.get("GEMINI_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = os.environ["GEMINI_API_KEY"]

assert os.environ.get("GOOGLE_API_KEY"), "Set GOOGLE_API_KEY first (see the comment above)."
print("API key loaded.")


## 1. ReAct as a 2-node graph — `agent ↔ tools`

In Week 2 you hand-rolled the Thought→Action→Observation loop. In LangGraph it's two nodes and one prebuilt router: the **agent** (an LLM with tools bound) and **`ToolNode`** (runs whatever tools the agent called). `tools_condition` checks the last message — if it contains tool calls, go to `tools`; else end. The edge `tools → agent` closes the loop.


In [ ]:
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI


@tool
def calculator(expression: str) -> str:
    """Evaluate a basic arithmetic expression, e.g. '2026 - 2017'."""
    import re
    if not re.fullmatch(r"[0-9\s+\-*/().]+", expression.replace("^", "**")):
        return "invalid expression"
    return str(eval(expression.replace("^", "**")))


@tool
def get_year(event: str) -> str:
    """Return the publication year of a known ML paper."""
    table = {"transformer": "2017", "react": "2023", "gpt-3": "2020", "bert": "2018"}
    for k, v in table.items():
        if k in event.lower():
            return v
    return "unknown"


tools = [calculator, get_year]


class State(TypedDict):
    messages: Annotated[list, add_messages]


llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
llm_with_tools = llm.bind_tools(tools)


def agent(state: State) -> dict:
    return {"messages": [llm_with_tools.invoke(state["messages"])]}


def build_graph():
    g = StateGraph(State)
    g.add_node("agent", agent)
    g.add_node("tools", ToolNode(tools))
    g.add_edge(START, "agent")
    g.add_conditional_edges("agent", tools_condition)   # -> "tools" or END
    g.add_edge("tools", "agent")
    return g


In [ ]:
app = build_graph().compile()
out = app.invoke({"messages": [("user",
    "What year was the transformer paper published, and how many years before the ReAct paper?")]})
for m in out["messages"]:
    m.pretty_print()


**See the graph.** The ReAct loop as two nodes — `agent` ⇄ `tools` — with a conditional edge that exits when the model stops calling tools. `mem_app`, `hil_app` and `durable_app` below share this exact shape; only the `compile()` options differ.

In [ ]:
# --- See the graph this code builds: ReAct: agent ⇄ tools ---
# A compiled LangGraph app can draw its own topology: nodes are steps,
# edges are control flow. This picture IS the agent you just wired.
try:
    from IPython.display import Image, display
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception:
    # draw_mermaid_png needs network/graphviz; ASCII always works
    print(app.get_graph().draw_ascii())

That's the entire ReAct loop from Week 2, now a graph: `agent → tools → agent → … → END`. Same control flow, but every node is inspectable and — next — pausable and persistent.

## 2. Memory across turns — `MemorySaver` + `thread_id`

A bare graph is stateless: each `invoke` starts fresh. Add a **checkpointer** and a `thread_id`, and the graph **remembers** the conversation. Same `thread_id` = same memory; new id = clean slate.


In [ ]:
from langgraph.checkpoint.memory import MemorySaver

mem_app = build_graph().compile(checkpointer=MemorySaver())
cfg = {"configurable": {"thread_id": "scientist-1"}}

mem_app.invoke({"messages": [("user", "What year was BERT published?")]}, cfg)
# follow-up with a pronoun — only works if it remembered the first turn
follow = mem_app.invoke({"messages": [("user", "And how many years before the ReAct paper was that?")]}, cfg)
print(follow["messages"][-1].content)


The follow-up says "that" with no antecedent in the message — it resolves only because the checkpointer replayed the prior turn. Switch to a new `thread_id` and the same question fails: that's the memory boundary, made explicit.

## 3. Human-in-the-loop — pause before acting

For anything consequential (spending, emailing, writing to a DB) you want a human to approve the tool call first. Compile with **`interrupt_before=["tools"]`**: the graph runs up to the tool node, then **stops** and returns. You inspect the pending action, then resume by invoking with `None`.


In [ ]:
hil_app = build_graph().compile(checkpointer=MemorySaver(), interrupt_before=["tools"])
cfg = {"configurable": {"thread_id": "approve-1"}}

# runs agent, then PAUSES before executing the tool
hil_app.invoke({"messages": [("user", "Compute 2026 - 1998.")]}, cfg)

state = hil_app.get_state(cfg)
pending = state.values["messages"][-1]
print("PAUSED. Agent wants to call:", pending.tool_calls)

# a human approves -> resume by invoking with None (picks up from the checkpoint)
resumed = hil_app.invoke(None, cfg)
print("After approval:", resumed["messages"][-1].content)


You could also **edit** the state before resuming (change the tool input, or inject a rejection message). That pause-inspect-resume cycle is impossible with a plain loop — it needs the checkpointer to freeze state mid-graph.

## 4. Durable state — `SqliteSaver`

`MemorySaver` lives in RAM and dies with the kernel. `SqliteSaver` writes checkpoints to a file, so a conversation survives a restart (or a crash, or a redeploy). Same API — swap the checkpointer.


In [ ]:
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

conn = sqlite3.connect("checkpoints.sqlite", check_same_thread=False)
durable_app = build_graph().compile(checkpointer=SqliteSaver(conn))
cfg = {"configurable": {"thread_id": "persistent-1"}}

durable_app.invoke({"messages": [("user", "What year was GPT-3 published?")]}, cfg)
# the checkpoint is now on disk in checkpoints.sqlite — a fresh process with the same
# thread_id + db would continue this exact conversation.
print("checkpoint written. resuming the thread:")
print(durable_app.invoke({"messages": [("user", "Subtract that from 2026.")]}, cfg)["messages"][-1].content)


## 5. Async parallel fan-out / fan-in

When steps are independent — query three sources, run three analyses — run them **in parallel**. Fan out from one node to several, then fan back in. The trick is a **reducer** on the state field so parallel writes *accumulate* instead of clobbering each other (`operator.add` appends lists).


In [ ]:
import asyncio, operator
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, START, END


class FanState(TypedDict):
    topic: str
    findings: Annotated[list, operator.add]   # reducer: parallel nodes APPEND here


async def source_a(state: FanState) -> dict:
    await asyncio.sleep(0.2)
    return {"findings": [f"[A] background on {state['topic']}"]}


async def source_b(state: FanState) -> dict:
    await asyncio.sleep(0.2)
    return {"findings": [f"[B] recent results on {state['topic']}"]}


async def source_c(state: FanState) -> dict:
    await asyncio.sleep(0.2)
    return {"findings": [f"[C] open problems in {state['topic']}"]}


def synthesize(state: FanState) -> dict:
    return {"findings": state["findings"] + ["[synth] " + " | ".join(state["findings"])]}


fan = StateGraph(FanState)
for name, fn in [("a", source_a), ("b", source_b), ("c", source_c)]:
    fan.add_node(name, fn)
fan.add_node("synthesize", synthesize)
for name in ["a", "b", "c"]:
    fan.add_edge(START, name)        # fan-out: all three start together
    fan.add_edge(name, "synthesize") # fan-in: all converge on synthesize
fan.add_edge("synthesize", END)
fan_app = fan.compile()

result = await fan_app.ainvoke({"topic": "sepsis biomarkers", "findings": []})
for f in result["findings"]:
    print(f)


**See the graph.** Fan-out / fan-in: `START` dispatches all three sources in parallel, then every branch converges on `synthesize` (the reducer merges their results).

In [ ]:
# --- See the graph this code builds: Parallel fan-out / fan-in ---
# A compiled LangGraph app can draw its own topology: nodes are steps,
# edges are control flow. This picture IS the agent you just wired.
try:
    from IPython.display import Image, display
    display(Image(fan_app.get_graph().draw_mermaid_png()))
except Exception:
    # draw_mermaid_png needs network/graphviz; ASCII always works
    print(fan_app.get_graph().draw_ascii())

Three nodes ran concurrently (≈0.2 s total, not 0.6 s), then `synthesize` saw all three results because the reducer accumulated them. **Async benefit:** wall-clock for independent work. **Async challenge:** every node on the path must be async-safe, and shared state needs a reducer or parallel writes race.

---

## Recap

- **`ToolNode` + `tools_condition`** = Week 2's ReAct loop as two nodes.
- **`MemorySaver` + `thread_id`** = memory across turns.
- **`interrupt_before`** = pause for human approval, then resume with `invoke(None, cfg)`.
- **`SqliteSaver`** = durable checkpoints that survive a restart.
- **Async fan-out + a reducer** = parallel steps that accumulate safely.

Next (capstone): when one agent isn't enough — a **supervisor** delegating to **worker** agents, the shape behind GPT-Researcher, open_deep_research, and TriAgent.
